<h2>1. Trenowanie modelu z optymalnymi wartościami hiperparametrów na całym zbiorze treningowym</h2>
<p>W drodze walidacji krzyżowej optymalne okazały się następujące wartości hiperparametrów:</p>
<ul>
<li><code>learning_rate = 0.01</code></li>
<li><code>weight_decay = 1e-4</code></li>
<li><code>batch_size = 16</code></li>
<li><code>epochs = 5</code></li>
</ul>


<h3>1.1. Import bibliotek</h3>

In [ ]:
from __future__ import division, print_function

import os.path
import tarfile
from glob import glob
from os import makedirs
from shutil import move, rmtree

import pandas as pd
from google.colab import drive

from dxtorchx.logging import Logger
from dxtorchx.model_const import densenet121
from dxtorchx.training import (
    is_gpu,
    auc_roc_metric,
    evaluate_dataset,
    IMAGES_DIR,
    TEST_SET_FILE,
    CNN_ARCHITECTURE,
    TRAIN_VAL_SET_FILE,
    TrainDataset,
    TestDataset,
    NeuralNetwork,
)

<h3>1.2. Definicja stałych</h3>

In [ ]:
MODEL_VERSION = "RELEASE"  # LR=0.01, WD=1e-4, BS=16

BEST_LEARNING_RATE = 0.01
BEST_WEIGHT_DECAY = 1e-4
BEST_BATCH_SIZE = 16

# plik etykietami dla zbioru testowego
Y_TRUE_FILE = f"y_true_{CNN_ARCHITECTURE}_{MODEL_VERSION}.csv"

# plik z prawdopodobieństwami wystąpenia każdej klasy dla zbioru testowego
Y_PROBABILITY_FILE = f"y_probability_{CNN_ARCHITECTURE}_{MODEL_VERSION}.csv"

# plik z metrykami dla każdej klasy dla zbioru testowego
AUC_METRICS_FILE = f"auc_metrics_{CNN_ARCHITECTURE}_{MODEL_VERSION}.csv"

<h3>1.3. Montuje Dysk Google, ponieważ tam znajdują się archiwa danych</h3>

In [3]:
drive.mount("/content/drive")

Mounted at /content/drive


<h3>1.4. Kopiuje pliki CSV z opisem zbiorów, zbiór treningowy i testowy</h3>

In [4]:
!cp /content/drive/MyDrive/_DATA_/nih_chest_x_ray_dataset/{LABEL_FILE_PREFIX}*.csv .

In [5]:
!ls

drive  nih_labels_test.csv  nih_labels_train_val.csv  sample_data


<h3>1.5. Kopiuje archiwa TAR z obrazami (12 plików TAR)</h3>

In [6]:
!rsync -av /content/drive/MyDrive/_DATA_/nih_chest_x_ray_dataset/archives/ .

sending incremental file list
./
images_01.tar.gz
images_02.tar.gz
images_03.tar.gz
images_04.tar.gz
images_05.tar.gz
images_06.tar.gz
images_07.tar.gz
images_08.tar.gz
images_09.tar.gz
images_10.tar.gz
images_11.tar.gz
images_12.tar.gz

sent 45,090,869,369 bytes  received 247 bytes  90,634,913.80 bytes/sec
total size is 45,079,862,784  speedup is 1.00


<h3>1.6. Sprawdzam czy wszystkie archiwa są skopiowane na lokalny dysk CoLAB</h3>

In [7]:
!ls

drive		  images_04.tar.gz  images_08.tar.gz  images_12.tar.gz
images_01.tar.gz  images_05.tar.gz  images_09.tar.gz  nih_labels_test.csv
images_02.tar.gz  images_06.tar.gz  images_10.tar.gz  nih_labels_train_val.csv
images_03.tar.gz  images_07.tar.gz  images_11.tar.gz  sample_data


<h3>1.7. Rozpakowuje archiwa ze zdjeciami do katalogów, zgodnymi z nazwami archiwów: <code>images_01</code>, <code>images_02</code> itp.</h3>

In [8]:
archives = [
    "images_01.tar.gz",
    "images_02.tar.gz",
    "images_03.tar.gz",
    "images_04.tar.gz",
    "images_05.tar.gz",
    "images_06.tar.gz",
    "images_07.tar.gz",
    "images_08.tar.gz",
    "images_09.tar.gz",
    "images_10.tar.gz",
    "images_11.tar.gz",
    "images_12.tar.gz",
]


def extract_tar(archive):
    print(f"Extracting {archive}... ")
    tar = tarfile.open(f"{archive}")
    tar.extractall(archive.split(".")[0])
    tar.close()
    print("Done!")


for archive in archives:
    extract_tar(archive)

Extracting images_01.tar.gz... 
Done!
Extracting images_02.tar.gz... 
Done!
Extracting images_03.tar.gz... 
Done!
Extracting images_04.tar.gz... 
Done!
Extracting images_05.tar.gz... 
Done!
Extracting images_06.tar.gz... 
Done!
Extracting images_07.tar.gz... 
Done!
Extracting images_08.tar.gz... 
Done!
Extracting images_09.tar.gz... 
Done!
Extracting images_10.tar.gz... 
Done!
Extracting images_11.tar.gz... 
Done!
Extracting images_12.tar.gz... 
Done!


In [9]:
!ls

drive		  images_03.tar.gz  images_06.tar.gz  images_09.tar.gz	images_12.tar.gz
images_01	  images_04	    images_07	      images_10		nih_labels_test.csv
images_01.tar.gz  images_04.tar.gz  images_07.tar.gz  images_10.tar.gz	nih_labels_train_val.csv
images_02	  images_05	    images_08	      images_11		sample_data
images_02.tar.gz  images_05.tar.gz  images_08.tar.gz  images_11.tar.gz
images_03	  images_06	    images_09	      images_12


<h3>1.8. Przenosze wszystkie zdjęcia do jednego katalogu <code>images</code></h3>

In [10]:
if not os.path.exists(IMAGES_DIR):
    makedirs(IMAGES_DIR)

# Liczba zdjęć 112120
total = 0
for archive in archives:
    dir_name = f"{archive.split('.')[0]}/{IMAGES_DIR}"
    print(dir_name)
    images = glob(f"{dir_name}/*")
    print(len(images))
    total += len(images)
    for image in images:
        file_name = image.split("/")[-1]
        assert file_name.endswith(".png")
        move(image, str(os.path.join(IMAGES_DIR, file_name)))
print(f"Liczba zdjęć: {total}")

images_01/images
4999
images_02/images
10000
images_03/images
10000
images_04/images
10000
images_05/images
10000
images_06/images
10000
images_07/images
10000
images_08/images
10000
images_09/images
10000
images_10/images
10000
images_11/images
10000
images_12/images
7121
Liczba zdjęć: 112120


<h3>1.9. Potwierdzam, że w katalogu <code>images</code> są wszystkie zdjęcia</h3>

In [11]:
images = [img.split("/")[-1] for img in glob(f"{IMAGES_DIR}/*")]
print(f"Liczba plików obrazów: {len(images)}")

Liczba plików obrazów: 112120


<h3>1.10. Usuwam puste katalogi</h3>

In [12]:
for archive in archives:
    dir_name = archive.split(".")[0]
    if os.path.exists(dir_name):
        rmtree(dir_name)
print("Gotowe!")

Gotowe!


In [13]:
!ls

drive		  images_03.tar.gz  images_07.tar.gz  images_11.tar.gz		sample_data
images		  images_04.tar.gz  images_08.tar.gz  images_12.tar.gz
images_01.tar.gz  images_05.tar.gz  images_09.tar.gz  nih_labels_test.csv
images_02.tar.gz  images_06.tar.gz  images_10.tar.gz  nih_labels_train_val.csv


<h3>1.11. Weryfikacja zdjęć ze zbioru CSV vs pliki obrazów na dysku</h3>
<p>Sprawdzam czy zdjęcia wymienione w plikach <code>CSV</code> zbioru testowego, treningowego i walidującego oraz te w katalogu <code>images</code> się zgadzają.</p>

In [14]:
test_df = pd.read_csv(TEST_SET_FILE)
train_val_df = pd.read_csv(TRAIN_VAL_SET_FILE)
all_df = pd.concat([test_df, train_val_df])
print(f"Liczba zdjęć w zbiorze CSV: {len(all_df)}")
# Liczba zdjec musi się zgadzać z liczba wymienioną w pliku i musza to byc te same zdjecia
assert len(all_df) == len(images)
assert len(set(all_df["Image Index"]) & set(images)) == len(images)
print("Weryfikacja poprawna!")
patients = all_df["Patient ID"].unique()
print(f"Liczba pacjentów: {len(patients)}")
print("-" * 60)

Liczba zdjęć w zbiorze CSV: 112120
Weryfikacja poprawna!
Liczba pacjentów: 30805
------------------------------------------------------------


<h2>2. Uruchamiam trening modelu z wybranymi hiperparametrami</h2>

In [34]:
if is_gpu():
    Logger.log_number_of_gpu()

train_ds = TrainDataset(fold=None)
model_to_train = densenet121()
neural_network = NeuralNetwork(
    learning_rate=BEST_LEARNING_RATE,
    weight_decay=BEST_WEIGHT_DECAY,
    batch_size=BEST_BATCH_SIZE,
    model=model_to_train,
    train_dataset=train_ds,
    validation_dataset=None,
    fold=None,
    version=MODEL_VERSION)
best_model = neural_network.train_final_model()

# ewaluacja modelu na zbiorze testowym
model = densenet121()
model.load_state_dict(best_model)
Y_true, Y_probability = evaluate_dataset(
    trained_model=model,
    dataset=TestDataset(),
    batch_size=BEST_BATCH_SIZE,
)
# zapisuje orginalne labelki (Ground Truth)
Y_true.to_csv(str(os.path.join(neural_network.RESULTS_DIR, Y_TRUE_FILE)), index=False)
# zapisuje prawdopodobieństwa (network outputs)
Y_probability.to_csv(str(os.path.join(neural_network.RESULTS_DIR, Y_PROBABILITY_FILE)), index=False)

# obliczanie metryki AUC ROC
auc_roc_df = auc_roc_metric(Y_true, Y_probability)
# i zapis wyników do pliku CSV
auc_roc_df.to_csv(str(os.path.join(neural_network.RESULTS_DIR, AUC_METRICS_FILE)), index=False)

Number of available GPUs: 1
Selected model: DenseNet
----------------------------------------------------------------------
Epoch 1/4 | Learning rate: 0.01
PHASE[train]: epoch:[1] train_loss: [0.16118] dataset_size: 101170
Model checkpoint! best_val_loss=[0.1611814264424224], epoch=[1]
----------------------------------------------------------------------
Epoch 2/4 | Learning rate: 0.01
PHASE[train]: epoch:[2] train_loss: [0.15072] dataset_size: 101170
Model checkpoint! best_val_loss=[0.15071778247640857], epoch=[2]
----------------------------------------------------------------------
Epoch 3/4 | Learning rate: 0.01
PHASE[train]: epoch:[3] train_loss: [0.14680] dataset_size: 101170
Model checkpoint! best_val_loss=[0.1467961799660089], epoch=[3]
----------------------------------------------------------------------
Epoch 4/4 | Learning rate: 0.01
PHASE[train]: epoch:[4] train_loss: [0.14401] dataset_size: 101170
Model checkpoint! best_val_loss=[0.14400960034724397], epoch=[4]
---------

<h3>2.1 Sprawdzam czy wyniku treningu zapisały się na lokalnym dysku Colab</h3>

In [35]:
!ls {RESULTS_DIR}/

auc_metrics_DENSENET_121_RELEASE.csv  y_probability_DENSENET_121_RELEASE.csv
checkpoint_DENSENET_121_RELEASE.pth   y_true_DENSENET_121_RELEASE.csv


<h2>2.2 Tworzę katalogi na wyniki na swoim dysku Google oraz kopiuje wyniki ewaluacji modelu na zbiorze testowym zapisane podczas treningu:</h2>
<ul>
<li>wagi modelu (używane będą w aplikacji zwracającej predykcje)</li>
<li>wartości prawdopodobieństw dla wszystkich zdjęć ze zbioru testowego</li>
<li>metryki AUC dla każdej klasy w zbiorze testowym</li>
</ul>

In [36]:
# Tworzę katalogi na dysku Google
target_path = f"/content/drive/MyDrive/_DATA_/nih_chest_x_ray_dataset/{neural_network.RESULTS_DIR}"
rmtree(target_path, ignore_errors=True)
os.makedirs(target_path, exist_ok=True)

In [37]:
# Kopiuje do siebie wyniki predykcji + wagi modelu + log z treningu itp
!cp {RESULTS_DIR}/* /content/drive/MyDrive/_DATA_/nih_chest_x_ray_dataset/{RESULTS_DIR}